In [4]:
!pip install -q nflreadpy pyarrow

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import nflreadpy as nfl
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

CACHE = "/content/drive/MyDrive/pbp_2015_2025_day3.parquet"
KEEP = ["game_id", "season", "season_type", "home_team", "posteam",
        "game_seconds_remaining", "yardline_100", "total_home_score",
        "total_away_score", "result", "spread_line",
        "home_timeouts_remaining", "away_timeouts_remaining", "down", "ydstogo",
        "home_wp", "vegas_home_wp"]
try:
    pbp = pd.read_parquet(CACHE)
    print("loaded from Drive")
except FileNotFoundError:
    parts = [nfl.load_pbp([s]).select(KEEP).to_pandas() for s in range(2015, 2026)]
    pbp = pd.concat(parts, ignore_index=True)
    pbp.to_parquet(CACHE)
    print("downloaded and saved to Drive")
print("raw:", pbp.shape)

downloaded and saved to Drive
raw: (532376, 17)


In [7]:
# Step 2: label + Day 2 features
df = pbp[(pbp["season_type"] == "REG") & pbp["posteam"].notna() & (pbp["result"] != 0)].copy()
df["home_win"] = (df["result"] > 0).astype(int)

df["home_lead"] = df["total_home_score"] - df["total_away_score"]
df["time_left"] = df["game_seconds_remaining"]
df["home_has_ball"] = (df["posteam"] == df["home_team"]).astype(float)
df["home_yards_to_score"] = np.where(df["home_has_ball"] == 1,
                                     df["yardline_100"], 100 - df["yardline_100"])

# Step 3: new features
frac_left = (df["game_seconds_remaining"] / 3600).clip(0, 1)   # kickoff = 1, end = 0
df["lead_x_time"]   = df["home_lead"] * (1 - frac_left)
df["spread"]        = df["spread_line"]
df["spread_x_time"] = df["spread_line"] * frac_left
df["home_to"] = df["home_timeouts_remaining"]
df["away_to"] = df["away_timeouts_remaining"]
df["down"] = df["down"].fillna(0)
for d in [1, 2, 3, 4]:
    df[f"down_{d}"] = (df["down"] == d).astype(float)
df["ydstogo"] = df["ydstogo"].fillna(0)

BASE = ["home_lead", "time_left", "home_has_ball", "home_yards_to_score"]
STEPS = [
    ("+ lead_x_time",     ["lead_x_time"]),
    ("+ spread",          ["spread", "spread_x_time"]),
    ("+ timeouts",        ["home_to", "away_to"]),
    ("+ down & distance", ["down_1", "down_2", "down_3", "down_4", "ydstogo"]),
]
ALL = BASE + [f for _, fs in STEPS for f in fs]
df = df.dropna(subset=ALL + ["home_win"])
print("clean:", df.shape)

clean: (471027, 31)


In [8]:
train = df[df["season"] <= 2022]
test  = df[df["season"] >= 2023]
y_train, y_test = train["home_win"].to_numpy(), test["home_win"].to_numpy()

def prep(features):
    Xtr, Xte = train[features].to_numpy(float), test[features].to_numpy(float)
    mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0)          # TRAIN stats only
    return (Xtr - mu) / sd, (Xte - mu) / sd, mu, sd

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

print("train:", train.shape, " test:", test.shape)

train: (340216, 31)  test: (130811, 31)


In [9]:
rows = []
feats = list(BASE)
for name, extra in [("Day 2 baseline (4 features)", [])] + STEPS:
    feats = feats + extra
    Xtr, Xte, _, _ = prep(feats)
    m = LogisticRegression(penalty=None, max_iter=2000).fit(Xtr, y_train)
    p = m.predict_proba(Xte)[:, 1]
    rows.append((name, len(feats), round(log_loss(y_test, p), 4),
                 round(((p >= 0.5) == y_test).mean(), 3)))

table = pd.DataFrame(rows, columns=["step", "n_features", "test_log_loss", "accuracy"])
print(table.to_string(index=False))
print("baseline (always guess train win rate):",
      round(log_loss(y_test, np.full(len(y_test), y_train.mean())), 4))

                       step  n_features  test_log_loss  accuracy
Day 2 baseline (4 features)           4         0.4958     0.743
              + lead_x_time           5         0.4878     0.743
                   + spread           7         0.4541     0.777
                 + timeouts           9         0.4541     0.777
          + down & distance          14         0.4541     0.777
baseline (always guess train win rate): 0.6898


In [10]:
cheat = BASE + ["result"]          # final score ni feature ga pettam (CHEATING)
Xtr, Xte, _, _ = prep(cheat)
m = LogisticRegression(penalty=None, max_iter=2000).fit(Xtr, y_train)
p = m.predict_proba(Xte)[:, 1]
print("LEAKAGE demo with `result` as a feature -> accuracy",
      round(((p >= 0.5) == y_test).mean(), 3), "(too good to be true!)")

LEAKAGE demo with `result` as a feature -> accuracy 1.0 (too good to be true!)


In [11]:
Xtr, Xte, mu, sd = prep(ALL)       # all 14 features

def fit_logreg_l2(X, y, lam=0.0, lr=0.5, epochs=3000):
    n, d = X.shape
    w, b = np.zeros(d), 0.0
    for _ in range(epochs):
        p = sigmoid(X @ w + b)          # 1. predict
        error = p - y                   # 2. guess - truth
        dw = X.T @ error / n + 2 * lam * w   # 3. gradient + L2 penalty
        db = error.mean()                    #    bias ki penalty ledu
        w -= lr * dw                    # 4. downhill step
        b -= lr * db
    return w, b

w0, b0 = fit_logreg_l2(Xtr, y_train, lam=0)
sk = LogisticRegression(penalty=None, max_iter=3000).fit(Xtr, y_train)
print("max |scratch - sklearn| weight diff:", round(np.abs(w0 - sk.coef_[0]).max(), 4))

max |scratch - sklearn| weight diff: 0.0013


In [14]:
groups = train["game_id"].to_numpy()
gkf = GroupKFold(n_splits=5)

cv_results = {}
for lam in [0, 0.001, 0.01, 0.1, 1]:
    scores = []
    for tr_idx, va_idx in gkf.split(Xtr, y_train, groups):
        w, b = fit_logreg_l2(Xtr[tr_idx], y_train[tr_idx], lam=lam, epochs=1000)
        p_val = sigmoid(Xtr[va_idx] @ w + b)
        scores.append(log_loss(y_train[va_idx], p_val))
    cv_results[lam] = np.mean(scores)
    print(f"lambda={lam:<6} val log loss {np.mean(scores):.4f} ± {np.std(scores):.4f}")

best_lam = min(cv_results, key=cv_results.get)
print("best lambda:", best_lam)

lambda=0      val log loss 0.4512 ± 0.0139
lambda=0.001  val log loss 0.4514 ± 0.0136
lambda=0.01   val log loss 0.4551 ± 0.0124
lambda=0.1    val log loss 0.5003 ± 0.0099
lambda=1      val log loss 0.6204 ± 0.0060
best lambda: 0


In [15]:
w, b = fit_logreg_l2(Xtr, y_train, lam=best_lam)
p_test = sigmoid(Xte @ w + b)
print(f"FINAL test log loss: {log_loss(y_test, p_test):.4f}  "
      f"accuracy: {((p_test >= 0.5) == y_test).mean():.3f}")

FINAL test log loss: 0.4541  accuracy: 0.777


In [16]:
sub = np.random.default_rng(0).choice(len(Xtr), 60000, replace=False)   # saga slow, 60K sample

coef = {}
for pen in ["l1", "l2"]:
    m = LogisticRegression(penalty=pen, C=0.001, solver="saga", max_iter=3000)
    m.fit(Xtr[sub], y_train[sub])
    coef[pen] = m.coef_[0]
coef["scratch (best λ)"] = w

print(pd.DataFrame(coef, index=ALL).round(3).to_string())

                        l1     l2  scratch (best λ)
home_lead            0.769  0.855             0.597
time_left            0.000 -0.024             0.001
home_has_ball        0.084  0.172             0.196
home_yards_to_score -0.025 -0.112            -0.129
lead_x_time          0.944  0.903             1.618
spread               0.356  0.370             0.476
spread_x_time        0.247  0.314             0.266
home_to              0.000  0.033             0.010
away_to              0.000 -0.045            -0.031
down_1               0.000  0.001            -0.001
down_2               0.000  0.002            -0.002
down_3               0.000 -0.008            -0.000
down_4               0.000  0.010             0.006
ydstogo              0.000 -0.015            -0.003


In [17]:
def win_prob(lead, secs_left, has_ball, yards_to_score, spread,
             home_to=3, away_to=3, down=1, ydstogo=10):
    frac = secs_left / 3600
    x = {"home_lead": lead, "time_left": secs_left, "home_has_ball": has_ball,
         "home_yards_to_score": yards_to_score, "lead_x_time": lead * (1 - frac),
         "spread": spread, "spread_x_time": spread * frac,
         "home_to": home_to, "away_to": away_to,
         "down_1": float(down == 1), "down_2": float(down == 2),
         "down_3": float(down == 3), "down_4": float(down == 4), "ydstogo": ydstogo}
    v = (np.array([x[f] for f in ALL]) - mu) / sd      # same features, same TRAIN mu/sd
    return sigmoid(v @ w + b)

# first play: away ball at its own 25 -> home_yards_to_score = 100 - 75 = 25
print("Up 7, 2 min left, home ball midfield :", round(win_prob(7, 120, 1, 50, 0), 3))
print("First play, tied, spread 0           :", round(win_prob(0, 3600, 0, 25, 0), 3))
print("First play, tied, home favored by 7  :", round(win_prob(0, 3600, 0, 25, 7), 3))
print("First play, tied, home underdog by 7 :", round(win_prob(0, 3600, 0, 25, -7), 3))

pk = df[df["spread_line"].abs() <= 1].drop_duplicates("game_id")
print("Real home win rate, pick'em games    :", round(pk["home_win"].mean(), 3), f"({len(pk)} games)")

Up 7, 2 min left, home ball midfield : 0.871
First play, tied, spread 0           : 0.457
First play, tied, home favored by 7  : 0.714
First play, tied, home underdog by 7 : 0.221
Real home win rate, pick'em games    : 0.476 (208 games)
